<a href="https://colab.research.google.com/github/ikbalktlk-crypto/rezervasyonsis/blob/main/Rezervasyonsis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yapay Zeka Destekli Otel Rezervasyon Sistemi

Bu proje, doğal dil işleme (NLP) yetenekleri ve **Tool Calling (Araç Çağırma)** mimarisi kullanılarak geliştirilmiş bir otel rezervasyon asistanıdır. Sistem, kullanıcılarla  bir otel resepsiyonisti kimliğiyle iletişim kurar, dinamik veritabanı sorguları yapar, güvenli ödeme simülasyonları gerçekleştirir ve e-posta onayı gönderir.

**Kullanılan Temel Teknolojiler:**
* **Büyük Dil Modeli (LLM):** OpenAI (gpt-4o-mini) & Tool Calling API
* **Kullanıcı Arayüzü (UI/UX):** Streamlit (Özel Figma CSS Enjeksiyonu ve Dinamik Modallar ile)
* **Veritabanı Yönetimi:** SQLite & Pandas (Veri manipülasyonu)
* **Ağ & Dağıtım:** Cloudflare Tunnels

In [1]:
!pip install -q "pandas==2.2.2" "google-auth==2.47.0" "google-genai" openai streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 30.1 MB/s eta 0:00:00


In [2]:
import urllib
print("ARAYÜZE GİRİŞ ŞİFRENİZ (IP):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

ARAYÜZE GİRİŞ ŞİFRENİZ (IP): 34.16.229.122


## 1. Gerekli Kütüphanelerin Kurulumu ve Ağ Yapılandırması
Sistemin çalışabilmesi için veri analizi (`pandas`), veritabanı yönetimi (`sqlite3`), LLM entegrasyonu (`openai`) ve arayüz rendering işlemleri (`streamlit`) için gerekli Python kütüphaneleri çalışma ortamına indirilmektedir. Ayrıca, güvenlik ve ağ yapılandırma işlemleri için yerel IP adresi tespit edilmektedir.

In [3]:
import pandas as pd
import sqlite3
import os
from google.colab import drive


drive.mount('/content/drive')

dosya_yolu = '/content/drive/MyDrive/otel_sistemi(Odalar)_duzeltilmiş.csv'

try:

    try:
        df = pd.read_csv(dosya_yolu, sep=';', encoding='utf-8')
    except:
        df = pd.read_csv(dosya_yolu, sep=';', encoding='iso-8859-9')


    df.columns = [col.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '')
                  .replace('ı', 'i').replace('ş', 's').replace('ç', 'c')
                  .replace('ö', 'o').replace('ü', 'u').replace('ğ', 'g')
                  .replace('.', '').replace('?', '').replace('-', '_').replace('²', '2') for col in df.columns]

    if os.path.exists('otel_yonetim.db'):
        os.remove('otel_yonetim.db')

    conn = sqlite3.connect('otel_yonetim.db')
    df.to_sql('Odalar', conn, if_exists='replace', index=False)

    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Rezervasyonlar (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            rezervasyon_kodu TEXT UNIQUE,
            oda_no TEXT,
            misafir_adi TEXT,
            giris DATE,
            cikis DATE
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Kullanicilar (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            kullanici_adi TEXT UNIQUE,
            sifre TEXT
        )
    ''')


    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Degerlendirmeler (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            puan INTEGER,
            yorum TEXT,
            tarih DATETIME
        )
    ''')

    conn.commit()
    conn.close()
    print("--- 🟢 VERİTABANI BAŞARIYLA HAZIRLANDI (YENİ ODA ÖZELLİKLERİ VE PUANLAMA TABLOSU EKLENDİ) ---")

except FileNotFoundError:
    print("🔴 HATA: CSV dosyası Drive'da bulunamadı.")
except Exception as e:
    print(f"🔴 Beklenmeyen bir hata oluştu: {e}")

Mounted at /content/drive
--- 🟢 VERİTABANI BAŞARIYLA HAZIRLANDI (YENİ ODA ÖZELLİKLERİ VE PUANLAMA TABLOSU EKLENDİ) ---


## 2. Veri Entegrasyonu ve İlişkisel Veritabanı (SQLite) Mimarisi
Bu bölümde, sistemin dinamik hafızasını oluşturacak ilişkisel veritabanı kurulmaktadır.
1. **Veri Temizliği:** Google Drive üzerinden çekilen CSV dosyasındaki sütun isimleri, yazılım standartlarına uygun olacak şekilde Türkçe karakterlerden ve boşluklardan arındırılır.
2. **Tablo Optimizasyonu:** Temizlenen veriler SQLite veritabanına aktarılarak `Odalar` tablosu oluşturulur.
3. **İlişkisel Yapı:** Sistem otonomisi için `Rezervasyonlar`, yetkilendirme için `Kullanicilar` ve sistemin başarısını ölçmek amacıyla `Degerlendirmeler` isimli yeni tablolar SQL komutlarıyla inşa edilir.

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('otel_yonetim.db')


sorgu_suite = "SELECT Oda_No, Tip, Manzara, Fiyat_TL FROM Odalar WHERE Tip='Suite' LIMIT 2"
suite_odalar = pd.read_sql(sorgu_suite, conn)


sorgu_genel = "SELECT Oda_No, Kat, Bina_Blok, Klima FROM Odalar LIMIT 3" # Removed Mini_Buzdolabi
genel_odalar = pd.read_sql(sorgu_genel, conn)

print("--- SUITE ODA TESTİ ---")
print(suite_odalar)
print("\n--- GENEL DONANIM TESTİ ---")
print(genel_odalar)

conn.close()

--- SUITE ODA TESTİ ---
  Oda_No    Tip Manzara  Fiyat_TL
0  A-103  Suite   Deniz      2500
1  A-106  Suite   Bahçe      2350

--- GENEL DONANIM TESTİ ---
  Oda_No  Kat Bina_Blok Klima
0  A-101    1    A Blok   Var
1  A-102    1    A Blok   Var
2  A-103    1    A Blok   Var



Kurulan veritabanı mimarisinin ve aktarılan CSV verilerinin doğruluğunu test etmek amacıyla örnek SQL sorguları (`SELECT`) atılır. Bu adım, yapay zekanın ileride yapacağı filtreleme işlemlerinin (örn: 'Suite' odaları getirme) hatasız çalışmasını garanti altına alır.

In [6]:
%%writefile arayuz.py
import sqlite3
import pandas as pd
import json
import smtplib
import secrets
import time
from email.message import EmailMessage
from openai import OpenAI
from datetime import datetime
import streamlit as st

# --- 1. TASARIM VE ARAYÜZ AYARLARI ---
SISTEM_ISMI = "Rezervasyon Sistemi"
st.set_page_config(page_title=SISTEM_ISMI, layout="centered")

st.markdown("""
<style>
    .stApp { background-color: #0A0A0A !important; }
    header { background-color: #0A0A0A !important; }
    [data-testid="stSidebar"] { background-color: #1A1919 !important; border-right: 1px solid #FFFFFF !important; }
    .block-container { max-width: 900px !important; padding-top: 2rem !important; }

    .bubble-container-ai { display: flex; justify-content: flex-start; margin-bottom: 25px; }
    .figma-ai { background-color: transparent; border: none; padding: 10px 15px; color: #FFFFFF; max-width: 85%; line-height: 1.6; border-left: 2px solid #695F6D; }

    .bubble-container-user { display: flex; justify-content: flex-end; margin-bottom: 25px; }
    .figma-user { background-color: #1A1919; border: 1px solid #FFFFFF; border-radius: 30px; padding: 15px 25px; color: #FFFFFF; max-width: 85%; line-height: 1.6; }

    [data-testid="stChatInput"] { background-color: transparent !important; border: none !important; }
    [data-testid="stChatInput"] > div { background-color: #695F6D !important; border: 1px solid #FFFFFF !important; border-radius: 40px !important; padding: 2px 10px !important; }
    [data-testid="stChatInput"] > div * { background-color: transparent !important; border: none !important; }
    [data-testid="stChatInput"] textarea { color: #FFFFFF !important; }
    [data-testid="stChatInputSubmitButton"] { color: #FFFFFF !important; }
    h1, h2, h3, p, div, label { color: #ffffff !important; }
    .stChatInputContainer { padding-bottom: 20px; }
</style>
""", unsafe_allow_html=True)

with st.sidebar:
    st.header("Sistem Ayarları")
    api_key = st.text_input("OpenAI API Şifresi:", type="password")
    gmail_pass = st.text_input("Gmail Uygulama Şifresi:", type="password")

    # --- YENİ EKLENEN SIFIRLAMA BUTONU ---
    st.markdown("---")
    if st.button("🔄 Yeni Rezervasyon (Sıfırla)", use_container_width=True):
        # Hafızadaki tüm durumları temizle
        for key in ["mesaj_gecmisi", "islem_basarili", "degerlendirme_yapildi", "aktif_odeme"]:
            if key in st.session_state:
                del st.session_state[key]
        st.rerun() # Sayfayı anında yenile

if not api_key or not gmail_pass:
    st.info("Lütfen sistemi başlatmak için soldaki menüden şifrelerinizi girin ve Enter'a basın.")
    st.stop()

client = OpenAI(api_key=api_key)
GMAIL_ADRESIM = "ikbalkutluk2004@gmail.com"
GMAIL_UYGULAMA_SIFRESI = gmail_pass
bugun_tarih = datetime.now().strftime('%Y-%m-%d')

# --- ARAYÜZDE AÇILAN MODALLAR ---
@st.dialog("💳 Güvenli Ödeme Ekranı")
def odeme_modali():
    data = st.session_state.aktif_odeme

    st.markdown(f"""
<div style='text-align: center; margin-bottom: 15px;'>
<p style='font-size: 15px; color: #FFFFFF;'>
Yapılacak toplam işlem tutarı <b>{data['tutar_tl']} TL</b> kadardır.<br>
Ödemeyi onaylıyorsanız lütfen bilgileri girip tamam butonuna tıklayınız.
</p>
</div>
    """, unsafe_allow_html=True)

    kart_isim = st.text_input("Kartın Üzerinde Yazan İsim")
    kart_no = st.text_input("Kart Numarası (Test için 4242 ile başlamalı)", max_chars=19)

    col1, col2 = st.columns(2)
    with col1:
        tarih = st.text_input("Tarih (AA/YY)")
    with col2:
        cvv = st.text_input("Güvenlik Kodu", type="password", max_chars=3)

    st.markdown("<br>", unsafe_allow_html=True)
    col_btn1, col_btn2 = st.columns(2)

    with col_btn1:
        if st.button("İptal Et", use_container_width=True):
            st.session_state.mesaj_gecmisi.append({"role": "user", "content": "SİSTEM BİLGİSİ: Kullanıcı ödemeyi iptal etti."})
            st.session_state.aktif_odeme = None
            st.rerun()

    with col_btn2:
        if st.button("Tamam", type="primary", use_container_width=True):
            clean_card = str(kart_no).replace(" ", "")
            if not clean_card.startswith("4242"):
                st.error("Geçersiz kart. Lütfen test kartı (4242) kullanınız.")
            elif not kart_isim or not tarih or not cvv:
                st.error("Lütfen tüm alanları doldurunuz.")
            else:
                with st.spinner("Ödeme onaylanıyor..."):
                    conn = sqlite3.connect('otel_yonetim.db')
                    cursor = conn.cursor()
                    cursor.execute("SELECT id FROM Rezervasyonlar WHERE oda_no = ? AND (giris < ? AND cikis > ?)", (data['oda_no'], data['cikis'], data['giris']))
                    if cursor.fetchone():
                        st.error("Üzgünüz, bu oda işlemi tamamlarken başkası tarafından rezerve edildi.")
                        conn.close()
                    else:
                        pnr = rezervasyon_kodu_uret()
                        cursor.execute("INSERT INTO Rezervasyonlar (rezervasyon_kodu, oda_no, misafir_adi, giris, cikis) VALUES (?,?,?,?,?)",
                                       (pnr, data['oda_no'], data['misafir_adi'], data['giris'], data['cikis']))
                        conn.commit()
                        conn.close()

                        guvenli_mail_gonder(data['alici_mail'], data['misafir_adi'], data['oda_no'], data['giris'], data['cikis'], data['tutar_tl'], pnr)

                        sohbet_et(f"SİSTEM BİLGİSİ: Ödeme başarıyla alındı. PNR Kodunuz: {pnr}. Lütfen rezervasyonun onaylandığını söyle ve değerlendirmeye yönlendir.")

                        st.session_state.aktif_odeme = None
                        st.session_state.islem_basarili = True
                        st.rerun()

@st.dialog("🌟 Asistanı Değerlendirin")
def puanlama_modali():
    st.markdown("<p style='text-align: center; font-size: 16px;'>Rezervasyon işleminiz başarıyla tamamlandı! Size sunduğum hizmeti 5 yıldız üzerinden değerlendirir misiniz?</p>", unsafe_allow_html=True)
    puan = st.feedback("stars")
    if puan is not None:
        st.session_state.degerlendirme_yapildi = True
        st.success(f"Harika! {puan + 1} yıldız verdiniz. Geri bildiriminiz için çok teşekkür ederim!")
        time.sleep(2)
        st.rerun()

# --- 2. FONKSİYONLAR ---
def bos_oda_bul(giris_tarihi, cikis_tarihi, oda_tipi):
    if giris_tarihi < bugun_tarih: return {"hata": "Geçmiş tarih."}
    if cikis_tarihi <= giris_tarihi: return {"hata": "Hatalı tarih aralığı."}
    conn = sqlite3.connect('otel_yonetim.db')
    query = """
    SELECT o.Oda_No, o.Tip, o.Manzara, o.Balkon_Tipi, o.Fiyat_TL
    FROM Odalar o
    LEFT JOIN Rezervasyonlar r ON o.Oda_No = r.oda_no AND (r.giris < ? AND r.cikis > ?)
    WHERE o.Tip = ? AND r.id IS NULL LIMIT 3
    """
    df = pd.read_sql_query(query, conn, params=(cikis_tarihi, giris_tarihi, oda_tipi))
    conn.close()
    return df.to_dict(orient='records')

def rezervasyon_kodu_uret():
    return f"OTL-{datetime.now().strftime('%Y')}-{secrets.token_hex(2).upper()}"

def rezervasyon_sorgula(misafir_adi=None, rezervasyon_kodu=None):
    conn = sqlite3.connect('otel_yonetim.db')
    sorgu = "SELECT rezervasyon_kodu, oda_no, misafir_adi, giris, cikis FROM Rezervasyonlar WHERE 1=1"
    params = []
    if misafir_adi: sorgu += " AND misafir_adi LIKE ?"; params.append(f"%{misafir_adi}%")
    if rezervasyon_kodu: sorgu += " AND rezervasyon_kodu = ?"; params.append(rezervasyon_kodu)
    df = pd.read_sql_query(sorgu, conn, params=params)
    conn.close()
    return df.to_string(index=False) if not df.empty else "Uygun kayıt bulunamadı."

def rezervasyon_iptal(rezervasyon_kodu, misafir_adi):
    conn = sqlite3.connect('otel_yonetim.db')
    cursor = conn.cursor()
    cursor.execute("DELETE FROM Rezervasyonlar WHERE rezervasyon_kodu = ? AND misafir_adi = ?", (rezervasyon_kodu, misafir_adi))
    conn.commit()
    rows = cursor.rowcount
    conn.close()
    return "İşlem Başarılı" if rows > 0 else "Hata: Kayıt bulunamadı."

def guvenli_mail_gonder(alici_mail, misafir_adi, oda_no, giris, cikis, toplam_tutar, rezervasyon_kodu):
    msg = EmailMessage()
    msg['Subject'] = f"Otel Rezervasyon Onayınız (Kod: {rezervasyon_kodu})"
    msg['To'] = alici_mail
    msg['From'] = GMAIL_ADRESIM

    icerik = f"""Değerli Misafirimiz {misafir_adi},

Otelimize göstermiş olduğunuz ilgi ve tercihiniz için size en içten teşekkürlerimizi sunarız.

Rezervasyon işleminiz ve ödemeniz başarıyla tamamlanmış olup, konforunuz için odanız sizin adınıza rezerve edilmiştir. Konaklama detaylarınızı aşağıda bulabilirsiniz:

• Rezervasyon (PNR) Kodunuz : {rezervasyon_kodu}
• Tahsis Edilen Oda Numarası: {oda_no}
• Giriş Tarihi                : {giris}
• Çıkış Tarihi               : {cikis}
• Tahsil Edilen Toplam Tutar : {toplam_tutar} TL

Giriş günü sizi ağırlamaktan büyük mutluluk duyacağız. İptal, değişiklik veya ek talepleriniz için PNR kodunuzla birlikte bu e-posta üzerinden bizimle dilediğiniz zaman iletişime geçebilirsiniz.

Şimdiden keyifli, huzurlu ve unutulmaz bir konaklama dileriz.

Saygılarımızla,
Otel Yönetim Ekibi"""

    msg.set_content(icerik)
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(GMAIL_ADRESIM, GMAIL_UYGULAMA_SIFRESI)
            smtp.send_message(msg)
        return "Onay e-postası başarıyla gönderildi!"
    except Exception as e:
        return f"Mail sunucu hatası: {str(e)}"

def odeme_ekranini_goster(misafir_adi, oda_no, giris, cikis, tutar_tl, alici_mail):
    st.session_state.aktif_odeme = {
        "misafir_adi": misafir_adi,
        "oda_no": oda_no,
        "giris": giris,
        "cikis": cikis,
        "tutar_tl": tutar_tl,
        "alici_mail": alici_mail
    }
    return "SİSTEM BİLGİSİ: Ödeme ekranı açıldı. Kullanıcının formu doldurması bekleniyor."

tools = [
    {
        "type": "function",
        "function": {
            "name": "bos_oda_bul",
            "description": "Belirtilen tarihlerde oteldeki boş odaları filtreler.",
            "parameters": {
                "type": "object",
                "properties": {
                    "giris_tarihi": {"type": "string"}, "cikis_tarihi": {"type": "string"},
                    "oda_tipi": {"type": "string", "enum": ["Single", "Double", "Suite"]}
                }, "required": ["giris_tarihi", "cikis_tarihi", "oda_tipi"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "odeme_ekranini_goster",
            "description": "Kullanıcıdan KART BİLGİSİ ALMADAN, ödeme yapmak üzere ekranda güvenli bir pencere açar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "misafir_adi": {"type": "string"}, "oda_no": {"type": "string"},
                    "giris": {"type": "string"}, "cikis": {"type": "string"},
                    "tutar_tl": {"type": "number"}, "alici_mail": {"type": "string"}
                }, "required": ["misafir_adi", "oda_no", "giris", "cikis", "tutar_tl", "alici_mail"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rezervasyon_sorgula",
            "description": "Misafir adı veya PNR ile rezervasyon arar.",
            "parameters": {
                "type": "object",
                "properties": {"misafir_adi": {"type": "string"}, "rezervasyon_kodu": {"type": "string"}}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rezervasyon_iptal",
            "description": "Rezervasyonu siler.",
            "parameters": {
                "type": "object",
                "properties": {"rezervasyon_kodu": {"type": "string"}, "misafir_adi": {"type": "string"}},
                "required": ["rezervasyon_kodu", "misafir_adi"]
            }
        }
    }
]

# --- 3. SİSTEM HAFIZASI VE DURUM YÖNETİMİ ---
if "mesaj_gecmisi" not in st.session_state:
    st.session_state.mesaj_gecmisi = [
        {
            "role": "system",
            "content": f"""Sen lüks bir otelin son derece kibar, profesyonel ve detaylara önem veren asistanısın. Bugünün tarihi: {bugun_tarih}.

İŞLEM ADIMLARI (BU SIRAYI KESİNLİKLE ATLAMAK YASAKTIR):
1. İHTİYAÇ BELİRLEME: Müşteriden giriş-çıkış tarihi ve oda tipini nazikçe öğren. Giriş ve çıkış tarihlerine göre konaklanacak GECE SAYISINI hesapla.
2. ODA SUNUMU: 'bos_oda_bul' aracını çalıştır. Veritabanından gelen fiyatlar GECELİK fiyattır. Müşteriye odaları sunarken odaların Yatak Tipi, Büyüklüğü (m2) gibi tüm özelliklerini, Gecelik Fiyatı ve hesapladığın TOPLAM KONAKLAMA TUTARINI (Gece Sayısı x Gecelik Fiyat) açıkça belirt.
3. KİMLİK BİLGİSİ: Odayı seçtiğinde Ad-Soyad ve E-posta adresi talep et.
4. KESİN ÖDEME ADIMI (KRİTİK): Müşteriden KART BİLGİLERİNİ SOHBETTE ASLA İSTEME! Ödemeyi almak için sadece `odeme_ekranini_goster` aracını çalıştır. Bu araca 'tutar_tl' olarak gecelik fiyatı değil, hesapladığın TOPLAM tutarı gönder.
5. ONAY VE YÖNLENDİRME: Ödeme işlemi ekrandan başarıyla tamamlandığında sistem sana PNR kodunu bildirecektir. O zaman işlemi sonlandır ve şu cümleyi kur: "Rezervasyonunuz başarıyla tamamlandı ve onay e-postanız gönderildi. Şimdi ekranınıza gelecek olan pencereden hizmetimi yıldız vererek değerlendirirseniz çok mutlu olurum." """
        }
    ]

if "islem_basarili" not in st.session_state:
    st.session_state.islem_basarili = False
if "degerlendirme_yapildi" not in st.session_state:
    st.session_state.degerlendirme_yapildi = False
if "aktif_odeme" not in st.session_state:
    st.session_state.aktif_odeme = None

def sohbet_et(mesaj):
    st.session_state.mesaj_gecmisi.append({"role": "user", "content": mesaj})
    sayac = 0

    while sayac < 5:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=st.session_state.mesaj_gecmisi,
            tools=tools
        )
        response_message = response.choices[0].message

        if not response_message.tool_calls:
            st.session_state.mesaj_gecmisi.append({"role": "assistant", "content": response_message.content})
            return response_message.content

        st.session_state.mesaj_gecmisi.append(response_message)
        sayac += 1

        for tool_call in response_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            func_name = tool_call.function.name

            func_map = {
                "bos_oda_bul": bos_oda_bul,
                "odeme_ekranini_goster": odeme_ekranini_goster,
                "rezervasyon_sorgula": rezervasyon_sorgula,
                "rezervasyon_iptal": rezervasyon_iptal
            }

            try:
                res = func_map[func_name](**args)
            except Exception as e:
                res = {"hata": f"Sistem araç çalıştırma hatası: {str(e)}"}

            st.session_state.mesaj_gecmisi.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": func_name,
                "content": json.dumps(res) if isinstance(res, (dict, list)) else str(res)
            })

    return "Şu an sistemlerimizde anlık bir yoğunluk yaşanıyor, lütfen işleminizi tekrar dener misiniz?"

# --- 4. SOHBET PENCERESİ VE ANA EKRAN ---

st.markdown("""
<div style='margin-bottom: 40px; padding-top: 20px;'>
<h2 style='color: #FFFFFF; font-weight: 400; font-size: 26px; margin-bottom: 30px;'>Rezervasyon Sistemi</h2>
<div style='display: flex; gap: 20px; align-items: flex-start;'>

<div style='position: relative; width: 45px; height: 45px; flex-shrink: 0;'>
<div style='position: absolute; top: 12px; left: 0px; width: 24px; height: 24px; background-color: #48D1CC; border-radius: 50%; z-index: 2;'></div>
<div style='position: absolute; top: 0px; left: 16px; width: 14px; height: 14px; background-color: #E0FFFF; border-radius: 50%; z-index: 1;'></div>
<div style='position: absolute; top: 16px; left: 20px; width: 16px; height: 16px; background-color: #008080; border-radius: 50%; z-index: 0;'></div>
</div>

<div>
<p style='color: #FFFFFF; font-size: 16px; margin: 0 0 10px 0; font-weight: 500;'>
Sistemimize Hoşgeldiniz. Rezervasyon asistanın senin için burada!
</p>
<p style='color: #CCCCCC; font-size: 13px; line-height: 1.6; margin: 0 0 20px 0;'>
Rezervasyon oluşturma, sorgulama, konaklama hakkında bilgi alma, ödeme ve iptal gibi dilediğiniz bir çok işlemi buradan kolayca tamamlayabilirsiniz.<br>
Size en iyi şekilde yardımcı olabilmem için bilgi edinmek istediğiniz konuyu benimle paylaşın lütfen.
</p>
<hr style='border: none; border-top: 1px solid #FFFFFF; width: 350px; margin: 0; opacity: 0.5;'>
</div>
</div>
</div>
""", unsafe_allow_html=True)

# 4.2 SOHBET GEÇMİŞİ EKRANI
for msg in st.session_state.mesaj_gecmisi:
    role = msg.get("role", "") if isinstance(msg, dict) else getattr(msg, "role", "")
    content = msg.get("content", "") if isinstance(msg, dict) else getattr(msg, "content", "")

    if role == "user" and "SİSTEM BİLGİSİ:" not in content and content:
        st.markdown(f'<div class="bubble-container-user"><div class="figma-user">{content}</div></div>', unsafe_allow_html=True)
    elif role == "assistant" and content:
        st.markdown(f'<div class="bubble-container-ai"><div class="figma-ai">{content}</div></div>', unsafe_allow_html=True)

kullanici_girdisi = st.chat_input("Mesajınızı yazın...")

if kullanici_girdisi:
    st.markdown(f'<div class="bubble-container-user"><div class="figma-user">{kullanici_girdisi}</div></div>', unsafe_allow_html=True)

    with st.spinner("İşleniyor..."):
        yanit = sohbet_et(kullanici_girdisi)
        st.markdown(f'<div class="bubble-container-ai"><div class="figma-ai">{yanit}</div></div>', unsafe_allow_html=True)

# TETİKLEYİCİ PENCERELER
if st.session_state.aktif_odeme:
    odeme_modali()
elif st.session_state.islem_basarili and not st.session_state.degerlendirme_yapildi:
    puanlama_modali()

Overwriting arayuz.py


## 3. LLM Ajanının Geliştirilmesi ve Kullanıcı Arayüzü (UI/UX)
Projenin merkezini oluşturan bu modül, `arayuz.py` adında bir web uygulaması üretir. Mimari dört ana bacağı kapsamaktadır:

* **1. Özel UI/UX Tasarımı (Figma to Streamlit):** Streamlit'in varsayılan tasarımı, HTML/CSS enjeksiyonu ile tamamen özelleştirilmiştir.
* **2. Dinamik Pencere (Modal) Yönetimi:** Kullanıcı deneyimini artırmak için, ödeme bilgileri ve değerlendirme adımları sohbetin dışına taşınarak, `@st.dialog` yapısı ile ekranın ortasında açılan güvenli pop-up formlarına (Modal) dönüştürülmüştür.
* **3. LLM Ajanı ve Prompt Mühendisliği (Prompt Engineering):** Ajanın hafızasına kesin iş akış kuralları (Odayı bul -> Kullanıcıyı kaydet -> Modal üzerinden ödeme al -> Mail at) katı bir "Sistem Mesajı" olarak işlenmiş ve 'Prompt Injection' saldırılarına karşı sınırlandırılmıştır.
* **4. Tool Calling (Araç Çağırma) Fonksiyonları:** Yapay zekanın dış dünyayla etkileşime girmesi için; veritabanında boş oda arayan (`bos_oda_bul`), ödeme ekranını tetikleyen (`odeme_ekranini_goster`) ve SMTP üzerinden gerçek zamanlı e-posta gönderen (`guvenli_mail_gonder`) Python fonksiyonları tanımlanıp sisteme entegre edilmiştir.

In [ ]:
import os
import time

# Eski süreçleri temizle (DİKKAT: Colab'i çökerten 'node' kapatma komutu silindi!)
!pkill -f streamlit
!pkill -f cloudflared

# Streamlit'i başlat
print("🚀 Sistem başlatılıyor...")
os.system("streamlit run arayuz.py --server.port 8511 &>/content/logs.txt &")
time.sleep(3)

# Cloudflare Tünelini kur
print("🚇 Güvenli bağlantı açılıyor...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8511 &> /content/tunnel_logs.txt &

time.sleep(5)
print("\n✅ SİSTEM HAZIR! Aşağıdaki linke tıklayarak arayüze gidebilirsin:")
!grep -o 'https://.*\.trycloudflare.com' /content/tunnel_logs.txt

🚀 Sistem başlatılıyor...
🚇 Güvenli bağlantı açılıyor...

✅ SİSTEM HAZIR! Aşağıdaki linke tıklayarak arayüze gidebilirsin:
https://something-refers-personals-original.trycloudflare.com


## 4. Sistemin Canlıya Alınması (Deployment & Cloudflare Tunnels)

* Geliştirilen Streamlit web uygulaması Google Colab'in yerel
sunucusunda (localhost) arka planda başlatılır. Sisteme  güvenli erişim sağlayabilmek için `Cloudflare Tunnels` altyapısı kullanılarak HTTPS şifrelemeli geçici bir tünel bağlantısı oluşturulur.